In [2]:
#!/usr/bin/env python

"""
Build prefix-level datasets for remaining-time prediction from MuProMAC event logs.

For each input CSV (already warm-up trimmed and split into train/val/test):
- Keep only *complete* cases (should already be true, but we double-check).
- For each event in a case, compute:
    * prefix_index       : event position within the case
    * elapsed_time       : timestamp - case_start_time
    * remaining_time     : case_complete_time - timestamp
    * open_cases         : #cases that are active at this timestamp
                           (case_start_time <= t < case_complete_time)
    * timesincelastevent : time since previous event in the same case
    * activity_duration  : end_time - timestamp (0 if end_time missing)

Output: one CSV per input file with prefix-level rows.
"""

import os
import glob

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# CONFIG: change paths as needed
# ------------------------------------------------------------------
INPUT_FOLDER  = r"out/251110/event_logs_splits"   # your split logs
OUTPUT_FOLDER = r"out/251110/prefix_datasets"     # new folder for prefix datasets
# ------------------------------------------------------------------


def load_split_log(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # basic sanity
    for col in ["timestamp", "case_id", "status"]:
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")

    return df


def keep_complete_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to cases that:
    - have at least one COMPLETE event
    - and whose *first* event in the (already warm-up trimmed) log is START.
    (This matches the splitting script you used.)
    """
    complete_cases = set(df.loc[df["status"] == "COMPLETE", "case_id"].unique())

    first_events = (
        df.sort_values(["case_id", "timestamp"])
          .groupby("case_id")
          .head(1)
    )
    good_starts = set(
        first_events.loc[first_events["status"] == "START", "case_id"].unique()
    )

    valid_cases = complete_cases & good_starts
    return df[df["case_id"].isin(valid_cases)].copy()


def _add_open_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 'open_cases' column to df (one row per event).

    Definition:
      open_cases(t) = number of cases with
          case_start_time <= t < case_complete_time

    Implementation:
      - Build per-case [start_time, end_time].
      - Build cumulative counts of starts and ends.
      - For each event timestamp t:
            open_cases(t) = #starts <= t  -  #ends <= t
    """
    intervals = (
        df.groupby("case_id")[["case_start_time", "case_complete_time"]]
          .first()
          .reset_index()
          .rename(columns={
              "case_start_time": "start_time",
              "case_complete_time": "end_time",
          })
    )

    if len(intervals) == 0:
        df["open_cases"] = 0
        return df

    # All starts: sort by start_time, cumulative count
    starts = (
        intervals[["start_time"]]
        .sort_values("start_time")
        .reset_index(drop=True)
    )
    starts["n_starts"] = np.arange(1, len(starts) + 1, dtype=np.int64)

    # All ends: sort by end_time, cumulative count
    ends = (
        intervals[["end_time"]]
        .sort_values("end_time")
        .reset_index(drop=True)
    )
    ends["n_ends"] = np.arange(1, len(ends) + 1, dtype=np.int64)

    # Work on a timestamp-sorted copy of df
    df_sorted = df.sort_values("timestamp").reset_index(drop=True)

    # Merge cumulative starts: n_starts = #cases with start_time <= t
    df_sorted = pd.merge_asof(
        df_sorted,
        starts,
        left_on="timestamp",
        right_on="start_time",
        direction="backward",
    )

    # Merge cumulative ends: n_ends = #cases with end_time <= t
    df_sorted = pd.merge_asof(
        df_sorted,
        ends,
        left_on="timestamp",
        right_on="end_time",
        direction="backward",
    )

    # Replace NaNs with 0 (no starts/ends before this time)
    df_sorted["n_starts"] = df_sorted["n_starts"].fillna(0)
    df_sorted["n_ends"]   = df_sorted["n_ends"].fillna(0)

    # open_cases = #starts <= t - #ends <= t
    df_sorted["open_cases"] = (
        df_sorted["n_starts"] - df_sorted["n_ends"]
    ).astype(int)

    # Clean up helper columns (only drop if they exist)
    helper_cols = ["start_time", "end_time", "n_starts", "n_ends"]
    drop_cols = [c for c in helper_cols if c in df_sorted.columns]
    df_sorted = df_sorted.drop(columns=drop_cols)

    return df_sorted


def build_prefix_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    From an event log (only complete cases), build prefix-level data with
    remaining_time, open_cases, timesincelastevent, activity_duration.
    """

    # --- per-case times ---
    case_start = (
        df.groupby("case_id")["timestamp"]
          .min()
    )

    comp_times = (
        df[df["status"] == "COMPLETE"]
        .groupby("case_id")["timestamp"]
        .min()
    )

    # keep only cases that have both start and completion
    valid_cases = set(case_start.index) & set(comp_times.index)
    df = df[df["case_id"].isin(valid_cases)].copy()

    # map times back
    df["case_start_time"]    = df["case_id"].map(case_start)
    df["case_complete_time"] = df["case_id"].map(comp_times)

    # --- activity_duration (if end_time is available) ---
    if "end_time" in df.columns:
        tmp = df["end_time"] - df["timestamp"]
        tmp = tmp.fillna(0.0)
        df["activity_duration"] = tmp
    else:
        # if end_time is missing entirely, just set 0
        df["activity_duration"] = 0.0

    # --- add open_cases (global WIP) ---
    df = _add_open_cases(df)

    # sort within cases
    df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)

    # time since previous event in the same case
    df["timesincelastevent"] = (
        df.groupby("case_id")["timestamp"]
          .diff()
          .fillna(0.0)
    )

    # prefix index within case (1,2,3,...)
    df["prefix_index"] = df.groupby("case_id").cumcount() + 1

    # elapsed time since case start
    df["elapsed_time"] = df["timestamp"] - df["case_start_time"]

    # remaining time until completion
    df["remaining_time"] = df["case_complete_time"] - df["timestamp"]

    # optional: drop gateway-only rows if you don’t want them
    # df = df[df["status"] != "gateway"].copy()

    # Choose columns to keep (you can add/remove later)
    keep_cols = [
        "case_id",
        "timestamp",
        "activity",
        "resource",
        "status",
        "prefix_index",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
        "open_cases",
        "timesincelastevent",
        "activity_duration",
    ]

    # plus any meta columns if present
    for extra in ["scenario", "method", "l", "simulation_run", "process"]:
        if extra in df.columns:
            keep_cols.append(extra)

    return df[keep_cols]


def process_split_file(path: str):
    base = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== Building prefixes for {base} ===")

    df = load_split_log(path)
    print(f"  Input events: {len(df)}, cases: {df['case_id'].nunique()}")

    df = keep_complete_cases(df)
    print(f"  After complete-case filter: {len(df)} events, "
          f"{df['case_id'].nunique()} cases")

    prefix_df = build_prefix_dataset(df)
    print(f"  Prefix rows: {len(prefix_df)} "
          f"(cases: {prefix_df['case_id'].nunique()})")

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    out_path = os.path.join(OUTPUT_FOLDER, f"{base}_prefix.csv")
    prefix_df.to_csv(out_path, index=False)
    print(f"  Saved prefix dataset -> {out_path}")


def main():
    if not os.path.isdir(INPUT_FOLDER):
        raise SystemExit(f"Input folder not found: {INPUT_FOLDER}")

    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))
    if not files:
        raise SystemExit(f"No CSV files in {INPUT_FOLDER}")

    print(f"Found {len(files)} split log(s) to process.")
    for path in files:
        # skip already processed prefix files if you re-run
        if path.endswith("_prefix.csv"):
            continue
        process_split_file(path)


if __name__ == "__main__":
    main()
#!/usr/bin/env python

"""
Build prefix-level datasets for remaining-time prediction from MuProMAC event logs.

For each input CSV (already warm-up trimmed and split into train/val/test):
- Keep only *complete* cases (should already be true, but we double-check).
- For each event in a case, compute:
    * prefix_index       : event position within the case
    * elapsed_time       : timestamp - case_start_time
    * remaining_time     : case_complete_time - timestamp
    * open_cases         : #cases that are active at this timestamp
                           (case_start_time <= t < case_complete_time)
    * timesincelastevent : time since previous event in the same case
    * activity_duration  : end_time - timestamp (0 if end_time missing)

Output: one CSV per input file with prefix-level rows.
"""

import os
import glob

import numpy as np
import pandas as pd


# ------------------------------------------------------------------
# CONFIG: change paths as needed
# ------------------------------------------------------------------
INPUT_FOLDER  = r"out/251110/event_logs_splits"   # your split logs
OUTPUT_FOLDER = r"out/251110/prefix_datasets"     # new folder for prefix datasets
# ------------------------------------------------------------------


def load_split_log(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # basic sanity
    for col in ["timestamp", "case_id", "status"]:
        if col not in df.columns:
            raise ValueError(f"{path} is missing required column: {col}")

    return df


def keep_complete_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Filter to cases that:
    - have at least one COMPLETE event
    - and whose *first* event in the (already warm-up trimmed) log is START.
    (This matches the splitting script you used.)
    """
    complete_cases = set(df.loc[df["status"] == "COMPLETE", "case_id"].unique())

    first_events = (
        df.sort_values(["case_id", "timestamp"])
          .groupby("case_id")
          .head(1)
    )
    good_starts = set(
        first_events.loc[first_events["status"] == "START", "case_id"].unique()
    )

    valid_cases = complete_cases & good_starts
    return df[df["case_id"].isin(valid_cases)].copy()


def _add_open_cases(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add 'open_cases' column to df (one row per event).

    Definition:
      open_cases(t) = number of cases with
          case_start_time <= t < case_complete_time

    Implementation:
      - Build per-case [start_time, end_time].
      - Build cumulative counts of starts and ends.
      - For each event timestamp t:
            open_cases(t) = #starts <= t  -  #ends <= t
    """
    intervals = (
        df.groupby("case_id")[["case_start_time", "case_complete_time"]]
          .first()
          .reset_index()
          .rename(columns={
              "case_start_time": "start_time",
              "case_complete_time": "end_time",
          })
    )

    if len(intervals) == 0:
        df["open_cases"] = 0
        return df

    # All starts: sort by start_time, cumulative count
    starts = (
        intervals[["start_time"]]
        .sort_values("start_time")
        .reset_index(drop=True)
    )
    starts["n_starts"] = np.arange(1, len(starts) + 1, dtype=np.int64)

    # All ends: sort by end_time, cumulative count
    ends = (
        intervals[["end_time"]]
        .sort_values("end_time")
        .reset_index(drop=True)
    )
    ends["n_ends"] = np.arange(1, len(ends) + 1, dtype=np.int64)

    # Work on a timestamp-sorted copy of df
    df_sorted = df.sort_values("timestamp").reset_index(drop=True)

    # Merge cumulative starts: n_starts = #cases with start_time <= t
    df_sorted = pd.merge_asof(
        df_sorted,
        starts,
        left_on="timestamp",
        right_on="start_time",
        direction="backward",
    )

    # Merge cumulative ends: n_ends = #cases with end_time <= t
    df_sorted = pd.merge_asof(
        df_sorted,
        ends,
        left_on="timestamp",
        right_on="end_time",
        direction="backward",
    )

    # Replace NaNs with 0 (no starts/ends before this time)
    df_sorted["n_starts"] = df_sorted["n_starts"].fillna(0)
    df_sorted["n_ends"]   = df_sorted["n_ends"].fillna(0)

    # open_cases = #starts <= t - #ends <= t
    df_sorted["open_cases"] = (
        df_sorted["n_starts"] - df_sorted["n_ends"]
    ).astype(int)

    # Clean up helper columns (only drop if they exist)
    helper_cols = ["start_time", "end_time", "n_starts", "n_ends"]
    drop_cols = [c for c in helper_cols if c in df_sorted.columns]
    df_sorted = df_sorted.drop(columns=drop_cols)

    return df_sorted


def build_prefix_dataset(df: pd.DataFrame) -> pd.DataFrame:
    """
    From an event log (only complete cases), build prefix-level data with
    remaining_time, open_cases, timesincelastevent, activity_duration.
    """

    # --- per-case times ---
    case_start = (
        df.groupby("case_id")["timestamp"]
          .min()
    )

    comp_times = (
        df[df["status"] == "COMPLETE"]
        .groupby("case_id")["timestamp"]
        .min()
    )

    # keep only cases that have both start and completion
    valid_cases = set(case_start.index) & set(comp_times.index)
    df = df[df["case_id"].isin(valid_cases)].copy()

    # map times back
    df["case_start_time"]    = df["case_id"].map(case_start)
    df["case_complete_time"] = df["case_id"].map(comp_times)

    # --- activity_duration (if end_time is available) ---
    if "end_time" in df.columns:
        tmp = df["end_time"] - df["timestamp"]
        tmp = tmp.fillna(0.0)
        df["activity_duration"] = tmp
    else:
        # if end_time is missing entirely, just set 0
        df["activity_duration"] = 0.0

    # --- add open_cases (global WIP) ---
    df = _add_open_cases(df)

    # sort within cases
    df = df.sort_values(["case_id", "timestamp"]).reset_index(drop=True)

    # time since previous event in the same case
    df["timesincelastevent"] = (
        df.groupby("case_id")["timestamp"]
          .diff()
          .fillna(0.0)
    )

    # prefix index within case (1,2,3,...)
    df["prefix_index"] = df.groupby("case_id").cumcount() + 1

    # elapsed time since case start
    df["elapsed_time"] = df["timestamp"] - df["case_start_time"]

    # remaining time until completion
    df["remaining_time"] = df["case_complete_time"] - df["timestamp"]

    # optional: drop gateway-only rows if you don’t want them
    # df = df[df["status"] != "gateway"].copy()

    # Choose columns to keep (you can add/remove later)
    keep_cols = [
        "case_id",
        "timestamp",
        "activity",
        "resource",
        "status",
        "prefix_index",
        "elapsed_time",
        "remaining_time",
        "case_start_time",
        "case_complete_time",
        "open_cases",
        "timesincelastevent",
        "activity_duration",
    ]

    # plus any meta columns if present
    for extra in ["scenario", "method", "l", "simulation_run", "process"]:
        if extra in df.columns:
            keep_cols.append(extra)

    return df[keep_cols]


def process_split_file(path: str):
    base = os.path.splitext(os.path.basename(path))[0]
    print(f"\n=== Building prefixes for {base} ===")

    df = load_split_log(path)
    print(f"  Input events: {len(df)}, cases: {df['case_id'].nunique()}")

    df = keep_complete_cases(df)
    print(f"  After complete-case filter: {len(df)} events, "
          f"{df['case_id'].nunique()} cases")

    prefix_df = build_prefix_dataset(df)
    print(f"  Prefix rows: {len(prefix_df)} "
          f"(cases: {prefix_df['case_id'].nunique()})")

    os.makedirs(OUTPUT_FOLDER, exist_ok=True)
    out_path = os.path.join(OUTPUT_FOLDER, f"{base}_prefix.csv")
    prefix_df.to_csv(out_path, index=False)
    print(f"  Saved prefix dataset -> {out_path}")


def main():
    if not os.path.isdir(INPUT_FOLDER):
        raise SystemExit(f"Input folder not found: {INPUT_FOLDER}")

    files = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.csv")))
    if not files:
        raise SystemExit(f"No CSV files in {INPUT_FOLDER}")

    print(f"Found {len(files)} split log(s) to process.")
    for path in files:
        # skip already processed prefix files if you re-run
        if path.endswith("_prefix.csv"):
            continue
        process_split_file(path)


if __name__ == "__main__":
    main()


Found 24 split log(s) to process.

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_test ===
  Input events: 47885, cases: 1508
  After complete-case filter: 47885 events, 1508 cases
  Prefix rows: 47885 (cases: 1508)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_test_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_train ===
  Input events: 143021, cases: 4524
  After complete-case filter: 142404 events, 4491 cases
  Prefix rows: 142404 (cases: 4491)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_train_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_dedicated_C1_warm10_val ===
  Input events: 47885, cases: 1508
  After complete-case filter: 47885 events, 1508 cases
  Prefix rows: 47885 (cases: 1508)
  Saved prefix dataset -> out/251110/prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_val_prefix.csv

=== Building prefixes for log_FIFO_run0_EXP_

In [5]:
import pandas as pd

path = r"out\251110\prefix_datasets\log_FIFO_run0_EXP_dedicated_C1_warm10_train_prefix.csv"
df = pd.read_csv(path)
df.head()
df.columns
df.dtypes




case_id                 int64
timestamp             float64
activity               object
resource               object
status                 object
prefix_index            int64
elapsed_time          float64
remaining_time        float64
case_start_time       float64
case_complete_time    float64
open_cases              int64
timesincelastevent    float64
activity_duration     float64
method                 object
l                     float64
simulation_run          int64
process                object
dtype: object

In [6]:
g = df.groupby("case_id")

# 1) case_start_time = min(timestamp)
start_ok = (g["timestamp"].min() == g["case_start_time"].first()).all()

# 2) case_complete_time = max(timestamp) OR = COMPLETE event timestamp
end_ok = (g["timestamp"].max() == g["case_complete_time"].first()).all()

start_ok, end_ok


(True, True)

In [7]:
(df["elapsed_time"] >= 0).all(), (df["remaining_time"] >= 0).all()


(True, True)

In [8]:
last_events = df.sort_values(["case_id","timestamp"]).groupby("case_id").tail(1)
last_events["remaining_time"].describe()


count    4491.0
mean        0.0
std         0.0
min         0.0
25%         0.0
50%         0.0
75%         0.0
max         0.0
Name: remaining_time, dtype: float64

In [9]:
idx_ok = (g["prefix_index"].max() == g.size()).all()
idx_ok


True

In [10]:
# first event per case
first_ev = df.sort_values(["case_id","timestamp"]).groupby("case_id").head(1)
first_ev["timesincelastevent"].describe()

# any negative?
(df["timesincelastevent"] < 0).any()


False

In [11]:
df["open_cases"].describe()
df["open_cases"].min()


0

In [12]:
# Proportion of events with open_cases == 0
prop_zero = (df["open_cases"] == 0).mean()
print(f"Share of events with open_cases == 0: {prop_zero:.4%}")

# How many such events in absolute count
count_zero = (df["open_cases"] == 0).sum()
print(f"Number of events with open_cases == 0: {count_zero} / {len(df)}")

Share of events with open_cases == 0: 0.0014%
Number of events with open_cases == 0: 2 / 142404


In [13]:
sample = df.sample(5, random_state=0)[["timestamp","case_id","open_cases"]]
sample


,timestamp,case_id,open_cases
8493,3856.335197,1129,25
52334,8964.802715,2510,39
54263,9164.425109,2571,43
91543,13410.077853,3746,17
131755,17835.918918,5015,15


In [16]:
def inspect_event(df, idx):
    """
    Inspect event at dataframe index `idx`:
    compare stored open_cases with naive recomputation.
    """
    row = df.iloc[idx]
    t = row["timestamp"]
    open_stored = row["open_cases"]

    # naive recomputation using all cases in this file
    active_cases = df[
        (df["case_start_time"] <= t) & (df["case_complete_time"] > t)
    ]["case_id"].nunique()

    print(f"Index       : {idx}")
    print(f"case_id     : {row['case_id']}")
    print(f"timestamp t : {t}")
    print(f"stored open_cases : {open_stored}")
    print(f"naive active_cases: {active_cases}")

    # optional: show first few active case_ids
    active_ids = df[
        (df["case_start_time"] <= t) & (df["case_complete_time"] > t)
    ]["case_id"].unique()[:10]
    print(f"Some active case_ids at t: {active_ids}")

# Example: inspect 3 random events
inspect_event(df, 0)
inspect_event(df, 1000)
inspect_event(df, 5000)


Index       : 0
case_id     : 861
timestamp t : 3023.5450620917227
stored open_cases : 1
naive active_cases: 1
Some active case_ids at t: [861]
Index       : 1000
case_id     : 892
timestamp t : 3238.711805460381
stored open_cases : 37
naive active_cases: 37
Some active case_ids at t: [888 892 896 898 900 901 902 903 904 905]
Index       : 5000
case_id     : 1019
timestamp t : 3565.3267631339886
stored open_cases : 47
naive active_cases: 47
Some active case_ids at t: [ 978  993  994  995  997  998  999 1001 1002 1003]


In [17]:
import pandas as pd

path_raw = r"out\251110\event_logs\test\log_FIFO_run0_EXP_dedicated_C1.csv"
df_raw = pd.read_csv(path_raw)
print(df_raw.columns)


Index(['method', 'num_processes', 'simulation_run', 'timestamp', 'process',
       'l', 'status', 'case_id', 'activity', 'resource', 'end_time',
       'cycle_time', 'data', 'queue_start'],
      dtype='object')


In [18]:
path_split = r"out\251110\event_logs_splits\log_FIFO_run0_EXP_dedicated_C1_warm10_train.csv"
df_split = pd.read_csv(path_split)
print(df_split.columns)


Index(['method', 'num_processes', 'simulation_run', 'timestamp', 'process',
       'l', 'status', 'case_id', 'activity', 'resource', 'end_time',
       'cycle_time', 'data', 'queue_start'],
      dtype='object')
